# 🤖 Entrenamiento del Modelo con Scikit-Learn

**Objetivo:** Utilizar el dataset de resultados internacionales limpio para entrenar modelos de Machine Learning clásico (Clasificación). Queremos predecir el resultado de un partido: Victoria Local, Empate o Victoria Visitante.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Cargamos los datos desde el CSV limpio generado en el notebook de limpieza


## 1. Carga de Datos Limpios

In [ ]:
df = pd.read_csv('../datasets/processed/partidos_internacionales_limpios.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Ingeniería de Variables (Feature Engineering)
Vamos a crear características matemáticas para que los algoritmos puedan predecir. 
El resultado será nuestra variable `y` (target).

In [ ]:
# Variable objetivo: resultado del partido
# 2 = gana local, 1 = empate, 0 = gana visitante
def determinar_resultado(row):
    if row['home_score'] > row['away_score']:
        return 2
    elif row['home_score'] == row['away_score']:
        return 1
    else:
        return 0

df['resultado'] = df.apply(determinar_resultado, axis=1)

# Feature: estadísticas históricas de goles por equipo
df_home = df.groupby('home_team').agg(
    goles_favor_local=('home_score', 'mean'),
    goles_contra_local=('away_score', 'mean')
).reset_index().rename(columns={'home_team': 'equipo'})

df_away = df.groupby('away_team').agg(
    goles_favor_visit=('away_score', 'mean'),
    goles_contra_visit=('home_score', 'mean')
).reset_index().rename(columns={'away_team': 'equipo'})

estadisticas = pd.merge(df_home, df_away, on='equipo', how='outer').fillna(0)
estadisticas['ataque'] = (estadisticas['goles_favor_local'] + estadisticas['goles_favor_visit']) / 2
estadisticas['defensa'] = (estadisticas['goles_contra_local'] + estadisticas['goles_contra_visit']) / 2
print(estadisticas.head())

In [ ]:
# Integramos estas estadísticas de ataque y defensa por partido
df = df.merge(estadisticas[['equipo','ataque','defensa']], left_on='home_team', right_on='equipo', how='left')
df.rename(columns={'ataque': 'ataque_local', 'defensa': 'defensa_local'}, inplace=True)
df.drop(columns=['equipo'], inplace=True)

df = df.merge(estadisticas[['equipo','ataque','defensa']], left_on='away_team', right_on='equipo', how='left')
df.rename(columns={'ataque': 'ataque_visit', 'defensa': 'defensa_visit'}, inplace=True)
df.drop(columns=['equipo'], inplace=True)

# Otras features
df['es_neutral'] = df['neutral'].astype(int)
torneo_map = {'mundial': 4, 'eliminatoria': 3, 'copa_continental': 2, 'amistoso': 1, 'otro': 1}
df['peso_torneo'] = df['tipo_torneo'].map(torneo_map).fillna(1)

FEATURES = ['ataque_local', 'defensa_local', 'ataque_visit', 'defensa_visit', 'es_neutral', 'peso_torneo']
df_modelo = df[FEATURES + ['resultado', 'year']].dropna()
df_modelo.head()

## 3. División Train/Test (Temporal)
En vez de usar un `train_test_split` aleatorio, hacemos una división temporal estricta para evitar la filtración de datos del futuro.

In [ ]:
X = df_modelo[FEATURES]
y = df_modelo['resultado']
years = df_modelo['year']

# Entrenamiento hasta 2022, Prueba con 2023+
X_train = X[years < 2023]
X_test  = X[years >= 2023]
y_train = y[years < 2023]
y_test  = y[years >= 2023]

print(f'Train: {X_train.shape[0]} partidos')
print(f'Test: {X_test.shape[0]} partidos')

## 4. Entrenamiento y Comparativa de Modelos

In [ ]:
modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'{nombre} - Accuracy Test: {acc:.3f}')

## 5. Análisis del Mejor Modelo (Random Forest)

In [ ]:
rf = modelos['Random Forest']
y_pred_rf = rf.predict(X_test)

print(classification_report(y_test, y_pred_rf, target_names=['Visitante gana','Empate','Local gana']))

fig, ax = plt.subplots(figsize=(6,5))
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(cm, display_labels=['Visit.', 'Empate', 'Local'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de Confusión - Random Forest')
plt.show()

## 6. Importancia de Variables (Feature Importance)

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8,5))
importancias.plot(kind='barh', ax=ax, color='#3b82f6')
ax.set_title('Importancia de las Variables - Random Forest')
ax.set_xlabel('Importancia Relativa')
plt.tight_layout()
plt.show()

## 7. Predicción de Ejemplo

In [ ]:
def obtener_stats(equipo, df_stats):
    row = df_stats[df_stats['equipo'] == equipo]
    if len(row) == 0:
        return {'ataque': 1.0, 'defensa': 1.0}
    return {'ataque': row['ataque'].values[0], 'defensa': row['defensa'].values[0]}

stats_arg = obtener_stats('Argentina', estadisticas)
stats_fra = obtener_stats('France', estadisticas)

partido = pd.DataFrame([{
    'ataque_local': stats_arg['ataque'],
    'defensa_local': stats_arg['defensa'],
    'ataque_visit': stats_fra['ataque'],
    'defensa_visit': stats_fra['defensa'],
    'es_neutral': 1,
    'peso_torneo': 4
}])

proba = rf.predict_proba(partido)[0]
print('Probabilidades Argentina vs Francia (Cancha Neutral):')
print(f'Gana Francia: {proba[0]*100:.1f}%')
print(f'Empate:       {proba[1]*100:.1f}%')
print(f'Gana Argentina: {proba[2]*100:.1f}%')

## 8. Exportación del Modelo y Estadísticas

Para poder utilizar nuestro modelo entrenado y las estadísticas de ataque y defensa en la aplicación web, los exportamos usando la librería `joblib`. Esto nos permite guardar su estado y realizar predicciones directamente sin tener que volver a entrenar el modelo.

In [ ]:
import joblib

# Guardamos el modelo Random Forest entrenado
joblib.dump(rf, '../models/modelo_rf.pkl')
print("Modelo Random Forest guardado en models/modelo_rf.pkl")

# Guardamos las estadísticas calculadas de ataque y defensa
joblib.dump(estadisticas, '../models/estadisticas.pkl')
print("Estadísticas guardadas en models/estadisticas.pkl")